In [1]:
pip install tensorflow numpy pandas matplotlib opencv-python scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
import cv2

# Dataset path
DATASET_PATH = r"C:\Users\rauna\Desktop\Main_Project\datasets"

# Image size
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

# Data Augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

# Load train dataset
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training"
)

# Load validation dataset
val_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation"
)

print("Class Mapping:", train_generator.class_indices)

Found 5933 images belonging to 2 classes.
Found 1483 images belonging to 2 classes.
Class Mapping: {'ALL blood': 0, 'normal  blood': 1}


In [3]:
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, LSTM, TimeDistributed, BatchNormalization

# Define CNN-LSTM Hybrid Model
model = Sequential([
    # CNN Feature Extractor
    Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Flatten(),

    # Reshaping for LSTM
    Dense(256, activation='relu'),
    Dropout(0.5),
    
    # LSTM Layer
    tf.keras.layers.Reshape((1, 256)),  # Reshape for LSTM
    LSTM(128, return_sequences=False),

    # Output Layer
    Dense(1, activation='sigmoid')  # Binary Classification (Leukemia vs Normal)
])

# Compile Model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Summary
model.summary()

C:\Users\rauna\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 61, 61, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 1, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,714,177 (25.61 MB)

 Trainable params: 6,713,729 (25.61 MB)

 Non-trainable params: 448 (1.75 KB)

In [4]:
# Train Model
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20
)

# Save Model
model.save("binary.keras")


Epoch 1/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 312s 2s/step - accuracy: 0.9966 - loss: 0.0110 - val_accuracy: 0.9953 - val_loss: 0.0296
Epoch 2/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 90s 484ms/step - accuracy: 0.9997 - loss: 0.0017 - val_accuracy: 0.8159 - val_loss: 0.4492
Epoch 3/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 100s 536ms/step - accuracy: 0.9997 - loss: 0.0024 - val_accuracy: 0.5610 - val_loss: 3.7683
Epoch 4/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 119s 642ms/step - accuracy: 1.0000 - loss: 1.4795e-04 - val_accuracy: 0.9825 - val_loss: 0.0957
Epoch 5/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 111s 596ms/step - accuracy: 0.9983 - loss: 0.0061 - val_accuracy: 0.9906 - val_loss: 0.0311
Epoch 6/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 103s 551ms/step - accuracy: 0.9995 - loss: 0.0024 - val_accuracy: 0.7539 - val_loss: 1.8379
Epoch 7/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 100s 536ms/step - accuracy: 0.9997 - loss: 9.9482e-04 - val_accuracy: 0.9946 - val_loss: 0.0241
Epoch 8/20
186/186 ━━━━━━━━━━━━━━━━━━━━ 100s 539ms/step - accuracy: 0.99

In [4]:
from tensorflow.keras.models import load_model
import cv2
import numpy as np

# Load trained model
model = load_model("binary.keras")

# Load test image
def predict_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (128, 128))
    img = img / 255.0  # Normalize
    img = np.expand_dims(img, axis=0)  # Add batch dimension

    prediction = model.predict(img)
    if prediction[0] < 0.5:
        print("Prediction: Leukemia detected")
    else:
        print("Prediction: Normal blood cell")

# Example Usage
predict_image(r"C:\Users\rauna\Desktop\Main_Project\datasets\normal  blood\ig\IG_5887.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step
Prediction: Normal blood cell


In [14]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
import cv2

DATASET_PATH = r"C:\Users\rauna\Desktop\Main_Project\datasets\ALL blood"

# Image size
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

# Data Augmentation for training
train_multi_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

multi_train_generator = train_multi_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training"
)

multi_val_generator = train_multi_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation"
)

print("Class Mapping:", multi_train_generator.class_indices)
print("Number of classes:", multi_train_generator.num_classes)



Found 2607 images belonging to 4 classes.
Found 649 images belonging to 4 classes.
Class Mapping: {'Benign': 0, 'Early': 1, 'Pre': 2, 'Pro': 3}
Number of classes: 4


In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, LSTM, BatchNormalization, Reshape
import tensorflow as tf

num_classes = 4   

multi_model = Sequential([
    # CNN Feature Extractor
    Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2,2)),

    Flatten(),

    # Dense + Dropout
    Dense(256, activation='relu'),
    Dropout(0.5),

    # LSTM Layer
    Reshape((1, 256)),     # same structure as your binary model
    LSTM(128, return_sequences=False),

    # 🔥 MULTI-CLASS OUTPUT
    Dense(num_classes, activation='softmax')
])

multi_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',   # 🔥 changed
    metrics=['accuracy']
)

multi_model.summary()


C:\Users\rauna\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 61, 61, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 1, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       197,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,714,564 (25.61 MB)

 Trainable params: 6,714,116 (25.61 MB)

 Non-trainable params: 448 (1.75 KB)

In [17]:
# Train Model
multi_history = multi_model.fit(
    multi_train_generator,
    validation_data=multi_val_generator,
    epochs=20
)

# Save Model
multi_model.save("leukemia.keras")

Epoch 1/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 109s 1s/step - accuracy: 0.7649 - loss: 0.6711 - val_accuracy: 0.2465 - val_loss: 2.3519
Epoch 2/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 41s 502ms/step - accuracy: 0.8738 - loss: 0.3552 - val_accuracy: 0.3035 - val_loss: 2.6232
Epoch 3/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 43s 522ms/step - accuracy: 0.9275 - loss: 0.2335 - val_accuracy: 0.3035 - val_loss: 4.7961
Epoch 4/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 45s 544ms/step - accuracy: 0.9333 - loss: 0.1974 - val_accuracy: 0.3035 - val_loss: 4.7777
Epoch 5/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 47s 575ms/step - accuracy: 0.9486 - loss: 0.1549 - val_accuracy: 0.3482 - val_loss: 3.9040
Epoch 6/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 52s 633ms/step - accuracy: 0.9409 - loss: 0.1769 - val_accuracy: 0.5901 - val_loss: 1.4202
Epoch 7/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 48s 587ms/step - accuracy: 0.9547 - loss: 0.1383 - val_accuracy: 0.4792 - val_loss: 1.8235
Epoch 8/20
82/82 ━━━━━━━━━━━━━━━━━━━━ 75s 505ms/step - accuracy: 0.9547 - loss: 0.1341 - val_accura

In [20]:
from tensorflow.keras.models import load_model
import cv2
import numpy as np

# Load trained MULTI-CLASS model
model = load_model("leukemia.keras")

# Class labels (must match training folder names)
class_labels = ["Benign", "Early", "Pre", "Pro"]

def predict_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (128, 128))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    prediction = model.predict(img)
    class_index = np.argmax(prediction)
    confidence = np.max(prediction)

    print(f"Prediction: {class_labels[class_index]}")
    print(f"Confidence: {confidence * 100:.2f}%")

# Example Usage
predict_image(
    r"C:\Users\rauna\Desktop\Main_Project\datasets\ALL blood\Pro\WBC-Malignant-Pro-002.jpg"
)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step
Prediction: Benign
Confidence: 69.16%


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

# Image size
IMG_SIZE = (128, 128)

# Load models
binary_model = load_model("binary.keras")          # binary classifier
leukemia_model = load_model("leukemia.keras")      # multi-class subtype model

# Class labels (MUST match folder names)
leukemia_classes = ["Benign", "Early", "Pre", "Pro"]

def classify_image(image_path):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Step 1: Binary classification
    binary_pred = binary_model.predict(img_array)[0][0]

    if binary_pred < 0.5:
        return "Normal Blood Cell"

    # Step 2: Leukemia subtype classification
    subtype_pred = leukemia_model.predict(img_array)
    subtype_index = np.argmax(subtype_pred)

    return f"Leukemia Detected → Subtype: {leukemia_classes[subtype_index]}"

# Example usage
print(
    classify_image(
        r"C:\Users\rauna\Desktop\Main_Project\datasets\normal  blood\ig\IG_5887.jpg"
    )
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 502ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 230ms/step
Leukemia Detected → Subtype: Benign


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications import ResNet50
import matplotlib.pyplot as plt

# Dataset path
DATASET_PATH = r"C:\Users\rauna\Desktop\Main_Project\datasets\ALL blood"

# Image size
IMG_SIZE = (128, 128)
BATCH_SIZE = 16  # Reduced for better gradient updates
EPOCHS = 50

# ============== DATA AUGMENTATION ==============
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,  # Added for blood cells
    fill_mode='nearest',
    validation_split=0.2
)

# Validation data - only rescaling
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Load train dataset
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

# Load validation dataset
val_generator = val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

print("=" * 50)
print("Class Mapping:", train_generator.class_indices)
print("Number of classes:", train_generator.num_classes)
print("Training samples:", train_generator.samples)
print("Validation samples:", val_generator.samples)
print("=" * 50)

# Check class distribution
class_counts = {}
for class_name in train_generator.class_indices.keys():
    class_path = os.path.join(DATASET_PATH, class_name)
    class_counts[class_name] = len(os.listdir(class_path))
print("\nClass Distribution:")
for class_name, count in class_counts.items():
    print(f"{class_name}: {count} images")
print("=" * 50)

# ============== IMPROVED CNN MODEL (NO LSTM) ==============
# LSTM doesn't make sense for single image classification
# Using deeper CNN instead

num_classes = train_generator.num_classes

model = Sequential([
    # Block 1
    Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(128, 128, 3)),
    BatchNormalization(),
    Conv2D(32, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    # Block 2
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(64, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    # Block 3
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    # Block 4
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    Conv2D(256, (3,3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    # Classification Head
    Flatten(),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    
    # Output Layer
    Dense(num_classes, activation='softmax')
])

# Compile with better optimizer settings
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)

model.summary()

# ============== CALLBACKS ==============
callbacks = [
    # Stop training if validation loss doesn't improve
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Reduce learning rate when stuck
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    
    # Save best model
    ModelCheckpoint(
        'best_multiclass_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# ============== TRAIN MODEL ==============
print("\n" + "=" * 50)
print("Starting Training...")
print("=" * 50 + "\n")

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

# Save final model
model.save("leukemia_multiclass_improved.keras")
print("\nModel saved as 'leukemia_multiclass_improved.keras'")



In [ ]:
# ============== PLOT TRAINING HISTORY ==============
def plot_history(history):
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Accuracy
    axes[0, 0].plot(history.history['accuracy'], label='Train Accuracy')
    axes[0, 0].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[0, 0].set_title('Model Accuracy')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # Loss
    axes[0, 1].plot(history.history['loss'], label='Train Loss')
    axes[0, 1].plot(history.history['val_loss'], label='Val Loss')
    axes[0, 1].set_title('Model Loss')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Precision
    axes[1, 0].plot(history.history['precision'], label='Train Precision')
    axes[1, 0].plot(history.history['val_precision'], label='Val Precision')
    axes[1, 0].set_title('Model Precision')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Recall
    axes[1, 1].plot(history.history['recall'], label='Train Recall')
    axes[1, 1].plot(history.history['val_recall'], label='Val Recall')
    axes[1, 1].set_title('Model Recall')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
    print("\nTraining plots saved as 'training_history.png'")
    plt.show()

plot_history(history)

# ============== EVALUATION ==============
print("\n" + "=" * 50)
print("Final Evaluation on Validation Set:")
print("=" * 50)
results = model.evaluate(val_generator)
print(f"\nValidation Loss: {results[0]:.4f}")
print(f"Validation Accuracy: {results[1]:.4f}")
print(f"Validation Precision: {results[2]:.4f}")
print(f"Validation Recall: {results[3]:.4f}")